# GAM for Classification with Non-linear Boundaries

**Topics:** Non-linear Classification, GAM, Smooth Terms

## Overview

Extend binary classification with smooth non-linear relationships using GAM.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.models.gam import fit_additive_gam, SmoothTerm
from aurora.validation.metrics import accuracy_score, roc_auc
from sklearn.metrics import roc_curve

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Non-linear Classification Data

In [ ]:
n = 400
x1 = np.random.uniform(-3, 3, n)
x2 = np.random.uniform(-3, 3, n)

# Non-linear decision boundary (circular + sinusoidal)
log_odds = (
    -1  # baseline
    - 0.5 * (x1**2 + x2**2 - 4)  # circular boundary
    + 1.5 * np.sin(x1 * 1.5)  # sinusoidal component
)

prob = 1 / (1 + np.exp(-log_odds))
y = np.random.binomial(1, prob)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y, 'prob_true': prob})

print(f"Generated {n} observations")
print(f"Class balance: {y.mean():.2f}")

# Visualize
plt.figure(figsize=(10, 8))
plt.scatter(df[df['y']==0]['x1'], df[df['y']==0]['x2'], 
            c='blue', alpha=0.6, s=50, label='Class 0', edgecolor='k', linewidth=0.5)
plt.scatter(df[df['y']==1]['x1'], df[df['y']==1]['x2'], 
            c='red', alpha=0.6, s=50, label='Class 1', edgecolor='k', linewidth=0.5)
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Non-linear Classification Data')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Compare: Logistic Regression vs GAM

In [ ]:
# 1. Linear logistic regression
X_linear = np.column_stack([np.ones(n), x1, x2])
result_logistic = fit_glm(X=X_linear, y=y, family='binomial')
prob_logistic = result_logistic.predict(X_linear)
y_pred_logistic = (prob_logistic >= 0.5).astype(int)

print("Logistic Regression:")
print(f"  Accuracy: {accuracy_score(y, y_pred_logistic):.3f}")
print(f"  ROC-AUC: {roc_auc(y, prob_logistic):.3f}")

# 2. GAM approach: Create smooth basis manually and use GLM
# This approach manually creates smooth bases and fits with GLM
from aurora.smoothing.splines.cubic import CubicSplineBasis

# Create knots for x1 and x2
knots_x1 = np.linspace(x1.min(), x1.max(), 10)
knots_x2 = np.linspace(x2.min(), x2.max(), 10)

# Create basis functions
basis_x1 = CubicSplineBasis(knots=knots_x1)
basis_x2 = CubicSplineBasis(knots=knots_x2)

# Build design matrix with smooth terms
X_smooth1 = basis_x1.basis_matrix(x1)
X_smooth2 = basis_x2.basis_matrix(x2)

# Combine: intercept + smooth basis for x1 + smooth basis for x2
X_gam = np.column_stack([np.ones(n), X_smooth1, X_smooth2])

# Fit with GLM (binomial family)
result_gam = fit_glm(X=X_gam, y=y, family='binomial')
prob_gam = result_gam.predict(X_gam)
y_pred_gam = (prob_gam >= 0.5).astype(int)

print("\nGAM (Smooth basis with GLM):")
print(f"  Accuracy: {accuracy_score(y, y_pred_gam):.3f}")
print(f"  ROC-AUC: {roc_auc(y, prob_gam):.3f}")
print(f"  Number of parameters: {X_gam.shape[1]}")

## Visualize Decision Boundaries

In [ ]:
# Create grid
x1_range = np.linspace(-3, 3, 100)
x2_range = np.linspace(-3, 3, 100)
x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)

# Logistic predictions
X_grid_linear = np.column_stack([
    np.ones(x1_grid.size),
    x1_grid.ravel(),
    x2_grid.ravel()
])
prob_grid_logistic = result_logistic.predict(X_grid_linear).reshape(x1_grid.shape)

# GAM predictions
X_grid_smooth1 = basis_x1.basis_matrix(x1_grid.ravel())
X_grid_smooth2 = basis_x2.basis_matrix(x2_grid.ravel())
X_grid_gam = np.column_stack([np.ones(x1_grid.size), X_grid_smooth1, X_grid_smooth2])
prob_grid_gam = result_gam.predict(X_grid_gam).reshape(x1_grid.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, prob_grid, title in zip(axes, 
                                 [prob_grid_logistic, prob_grid_gam],
                                 ['Logistic Regression', 'GAM']):
    contour = ax.contourf(x1_grid, x2_grid, prob_grid, levels=20, 
                          cmap='RdBu_r', alpha=0.6, vmin=0, vmax=1)
    ax.contour(x1_grid, x2_grid, prob_grid, levels=[0.5], 
               colors='black', linewidths=3, linestyles='--')
    ax.scatter(df[df['y']==0]['x1'], df[df['y']==0]['x2'], 
               c='blue', marker='o', s=30, edgecolor='k', alpha=0.7, linewidth=0.5)
    ax.scatter(df[df['y']==1]['x1'], df[df['y']==1]['x2'], 
               c='red', marker='X', s=30, edgecolor='k', alpha=0.7, linewidth=0.5)
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.set_title(title)
    plt.colorbar(contour, ax=ax, label='P(Y=1)')

plt.tight_layout()
plt.show()

print("\nGAM captures non-linear boundary much better!")
print("Logistic regression limited to linear boundary")

## ROC Comparison

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y, prob_logistic)
fpr_gam, tpr_gam, _ = roc_curve(y, prob_gam)

plt.figure(figsize=(8, 8))
plt.plot(fpr_log, tpr_log, lw=2, label=f'Logistic (AUC={roc_auc(y, prob_logistic):.3f})')
plt.plot(fpr_gam, tpr_gam, lw=2, label=f'GAM (AUC={roc_auc(y, prob_gam):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves: Logistic vs GAM')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Key takeaway:** Use GAM when linear boundaries are insufficient!

**Next:** See `03_count_data/01_poisson_regression.ipynb` for count outcomes